- Delete Duplicate
- Column Schema Evolution
- DataTypes Handleing / Type casting
- Null Records Handling
- Triming
- Case Conversion
- Maintain Order of Data

In [0]:
dbutils.widgets.dropdown(name = "environment", defaultValue= "dev",choices= ["dev","prd","qa"],label = "select Environment")
env = dbutils.widgets.get("environment")
# print(env)

In [0]:
silverTablName = f"saleslake_{env}.silver_{env}.cleanedCustomer"
# print(silverTablName)
bronzeTablName = f"saleslake_{env}.bronze_{env}.rawCustomer"
# print(bronzeTablName)
srcFileLoc=f"/Volumes/saleslake_{env}/silver_{env}/vol_saleslake_src_files_{env}/daily_customer/"
# print(srcFileLoc)

In [0]:
spark.sql(f"""INSERT INTO {silverTablName}
SELECT DISTINCT   -- customer_id,customer_name,email,phone,address,city,state,country,zip_code,segment
    CAST(TRIM(customer_id) AS INTEGER) as customer_id ,
    UPPER(TRIM(customer_name)) as customer_name,
    UPPER(TRIM(email)) as email,
    UPPER(TRIM(phone)) as phone,
    UPPER(TRIM(address)) as address,
    UPPER(TRIM(city)) as city,
    UPPER(TRIM(state)) as state,
    UPPER(TRIM(country)) as country,
    UPPER(TRIM(zip_code)) as zip_code,
    UPPER(TRIM(segment)) as segment,
    CURRENT_TIMESTAMP() as ingest_ts
FROM {bronzeTablName}
WHERE ingest_ts > (
                    SELECT coalesce(MAX(ingest_ts),TO_DATE('1990-01-01','yyyy-MM-dd')) 
                    FROM {silverTablName}
                    )
ORDER BY CAST(TRIM(customer_id) AS INTEGER)
""")


In [0]:
%sql

-- SELECT * FROM saleslake_dev.bronze_dev.rawCustomer;
-- SELECT * FROM saleslake_dev.gold_dev.refinedcustomer;
-- SELECT * FROM saleslake_dev.silver_dev.cleanedCustomer;

-- TRUNCATE TABLE saleslake_dev.bronze_dev.rawCustomer;
-- TRUNCATE TABLE saleslake_dev.gold_dev.refinedcustomer;
-- TRUNCATE TABLE saleslake_dev.silver_dev.cleanedCustomer;
